In [1]:
import findspark
findspark.init('C:\\dev\\tools\\spark-2.4.4-bin-hadoop2.7')

import configparser
from datetime import datetime
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.functions import year, month, dayofmonth, hour, weekofyear, date_format

spark = SparkSession \
        .builder \
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:2.7.0") \
        .getOrCreate()

In [2]:
# get filepath to song data file
song_data = "data/song-data/*/*/*/*.json"
song_df = spark.read.json(song_data)
song_df.printSchema()

root
 |-- artist_id: string (nullable = true)
 |-- artist_latitude: double (nullable = true)
 |-- artist_location: string (nullable = true)
 |-- artist_longitude: double (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- num_songs: long (nullable = true)
 |-- song_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- year: long (nullable = true)



In [3]:
# extract columns to create songs table
songs_table = song_df.select("song_id", "title", "artist_id", "year", "duration")

In [4]:
songs_table.take(5)

[Row(song_id='SOBAYLL12A8C138AF9', title='Sono andati? Fingevo di dormire', artist_id='ARDR4AC1187FB371A1', year=0, duration=511.16363),
 Row(song_id='SOOLYAZ12A6701F4A6', title='Laws Patrolling (Album Version)', artist_id='AREBBGV1187FB523D2', year=0, duration=173.66159),
 Row(song_id='SOBBUGU12A8C13E95D', title='Setting Fire to Sleeping Giants', artist_id='ARMAC4T1187FB3FA4C', year=2004, duration=207.77751),
 Row(song_id='SOAOIBZ12AB01815BE', title='I Hold Your Hand In Mine [Live At Royal Albert Hall]', artist_id='ARPBNLO1187FB3D52F', year=2000, duration=43.36281),
 Row(song_id='SONYPOM12A8C13B2D7', title='I Think My Wife Is Running Around On Me (Taco Hell)', artist_id='ARDNS031187B9924F0', year=2005, duration=186.48771)]

In [6]:
# write songs table to parquet files partitioned by year and artist
songs_table.write.partitionBy("year", "artist_id").parquet("output/songs_out/")

In [7]:
# extract columns to create artists table
artists_table = song_df.select("artist_id", "artist_name", "artist_location", "artist_latitude", "artist_longitude") \
    .withColumnRenamed("artist_name", "name") \
    .withColumnRenamed("artist_location", "location") \
    .withColumnRenamed("artist_latitude", "latitude") \
    .withColumnRenamed("artist_longitude", "longitude")

In [8]:
artists_table.take(5)

[Row(artist_id='ARDR4AC1187FB371A1', name='Montserrat Caballé;Placido Domingo;Vicente Sardinero;Judith Blegen;Sherrill Milnes;Georg Solti', location='', latitude=None, longitude=None),
 Row(artist_id='AREBBGV1187FB523D2', name="Mike Jones (Featuring CJ_ Mello & Lil' Bran)", location='Houston, TX', latitude=None, longitude=None),
 Row(artist_id='ARMAC4T1187FB3FA4C', name='The Dillinger Escape Plan', location='Morris Plains, NJ', latitude=40.82624, longitude=-74.47995),
 Row(artist_id='ARPBNLO1187FB3D52F', name='Tiny Tim', location='New York, NY', latitude=40.71455, longitude=-74.00712),
 Row(artist_id='ARDNS031187B9924F0', name='Tim Wilson', location='Georgia', latitude=32.67828, longitude=-83.22295)]

In [9]:
# write artists table to parquet files
artists_table.write.parquet("output/artists_out")

In [10]:
# get filepath to log data file
log_data = "data/log-data/*.json"

# read log data file
log_df = spark.read.json(log_data)

In [11]:
log_df.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: double (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: double (nullable = true)
 |-- sessionId: long (nullable = true)
 |-- song: string (nullable = true)
 |-- status: long (nullable = true)
 |-- ts: long (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



In [12]:
log_df.take(5)

[Row(artist='Harmonia', auth='Logged In', firstName='Ryan', gender='M', itemInSession=0, lastName='Smith', length=655.77751, level='free', location='San Jose-Sunnyvale-Santa Clara, CA', method='PUT', page='NextSong', registration=1541016707796.0, sessionId=583, song='Sehr kosmisch', status=200, ts=1542241826796, userAgent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome/36.0.1985.125 Safari/537.36"', userId='26'),
 Row(artist='The Prodigy', auth='Logged In', firstName='Ryan', gender='M', itemInSession=1, lastName='Smith', length=260.07465, level='free', location='San Jose-Sunnyvale-Santa Clara, CA', method='PUT', page='NextSong', registration=1541016707796.0, sessionId=583, song='The Big Gundown', status=200, ts=1542242481796, userAgent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome/36.0.1985.125 Safari/537.36"', userId='26'),
 Row(artist='Train', auth='Logged In'

In [13]:
log_df.count()

8056

In [14]:
# filter by actions for song plays
log_df = log_df.filter(log_df.page == "NextSong")

In [15]:
log_df.count()

6820

In [16]:
# extract columns for users table    
users_table = log_df.select("userId", "firstName", "lastName", "gender", "level") \
    .dropDuplicates() \
    .withColumnRenamed("userId", "user_id") \
    .withColumnRenamed("firstName", "first_name") \
    .withColumnRenamed("lastName", "last_name")

In [17]:
users_table.take(5)

[Row(user_id='57', first_name='Katherine', last_name='Gay', gender='F', level='free'),
 Row(user_id='84', first_name='Shakira', last_name='Hunt', gender='F', level='free'),
 Row(user_id='22', first_name='Sean', last_name='Wilson', gender='F', level='free'),
 Row(user_id='52', first_name='Theodore', last_name='Smith', gender='M', level='free'),
 Row(user_id='80', first_name='Tegan', last_name='Levine', gender='F', level='paid')]

In [18]:
# write users table to parquet files
users_table.write.parquet("output/users_out")

In [19]:
from pyspark.sql.types import IntegerType

# create timestamp column from original timestamp column
get_timestamp = udf(lambda x: int(int(x) / 1000), IntegerType())
log_df = log_df.withColumn("timestamp", get_timestamp(col("ts")))

In [20]:
log_df.take(5)

[Row(artist='Harmonia', auth='Logged In', firstName='Ryan', gender='M', itemInSession=0, lastName='Smith', length=655.77751, level='free', location='San Jose-Sunnyvale-Santa Clara, CA', method='PUT', page='NextSong', registration=1541016707796.0, sessionId=583, song='Sehr kosmisch', status=200, ts=1542241826796, userAgent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome/36.0.1985.125 Safari/537.36"', userId='26', timestamp=1542241826),
 Row(artist='The Prodigy', auth='Logged In', firstName='Ryan', gender='M', itemInSession=1, lastName='Smith', length=260.07465, level='free', location='San Jose-Sunnyvale-Santa Clara, CA', method='PUT', page='NextSong', registration=1541016707796.0, sessionId=583, song='The Big Gundown', status=200, ts=1542242481796, userAgent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome/36.0.1985.125 Safari/537.36"', userId='26', timestamp=154224

In [21]:
from pyspark.sql.types import TimestampType

# create datetime column from original timestamp column
get_datetime = udf(lambda x: datetime.fromtimestamp(x / 1000), TimestampType())
log_df = log_df.withColumn("datetime", get_datetime(col("ts")))

In [22]:
log_df.take(5)

[Row(artist='Harmonia', auth='Logged In', firstName='Ryan', gender='M', itemInSession=0, lastName='Smith', length=655.77751, level='free', location='San Jose-Sunnyvale-Santa Clara, CA', method='PUT', page='NextSong', registration=1541016707796.0, sessionId=583, song='Sehr kosmisch', status=200, ts=1542241826796, userAgent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome/36.0.1985.125 Safari/537.36"', userId='26', timestamp=1542241826, datetime=datetime.datetime(2018, 11, 15, 1, 30, 26, 796000)),
 Row(artist='The Prodigy', auth='Logged In', firstName='Ryan', gender='M', itemInSession=1, lastName='Smith', length=260.07465, level='free', location='San Jose-Sunnyvale-Santa Clara, CA', method='PUT', page='NextSong', registration=1541016707796.0, sessionId=583, song='The Big Gundown', status=200, ts=1542242481796, userAgent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome

In [23]:
# extract columns to create time table
time_table = log_df.select("datetime"). \
    withColumnRenamed("datetime", "start_time"). \
    withColumn("hour", hour("start_time")). \
    withColumn("day", dayofmonth("start_time")). \
    withColumn("week", weekofyear("start_time")). \
    withColumn("month", month("start_time")). \
    withColumn("year", year("start_time")). \
    withColumn("weekday", date_format("start_time", "EEE"))

In [24]:
time_table.take(5)

[Row(start_time=datetime.datetime(2018, 11, 15, 1, 30, 26, 796000), hour=1, day=15, week=46, month=11, year=2018, weekday='Thu'),
 Row(start_time=datetime.datetime(2018, 11, 15, 1, 41, 21, 796000), hour=1, day=15, week=46, month=11, year=2018, weekday='Thu'),
 Row(start_time=datetime.datetime(2018, 11, 15, 1, 45, 41, 796000), hour=1, day=15, week=46, month=11, year=2018, weekday='Thu'),
 Row(start_time=datetime.datetime(2018, 11, 15, 4, 44, 9, 796000), hour=4, day=15, week=46, month=11, year=2018, weekday='Thu'),
 Row(start_time=datetime.datetime(2018, 11, 15, 6, 48, 55, 796000), hour=6, day=15, week=46, month=11, year=2018, weekday='Thu')]

In [26]:
# write time table to parquet files partitioned by year and month
time_table.write.partitionBy("year", "month").parquet("output/time_out/")

In [27]:
# read in song data to use for songplays table
song_df = spark.read.json(song_data)

In [28]:
song_df_valid = song_df.dropna(how = "any", subset = ["artist_name", "title"])
log_df_valid = log_df.dropna(how = "any", subset = ["artist", "song"])

In [29]:
print(song_df_valid.count())
print(log_df_valid.count())

71
6820


In [31]:
from pyspark.sql.functions import monotonically_increasing_id

# extract columns from joined song and log datasets to create songplays table 
songplays_table = log_df_valid.join(song_df, \
                                    [log_df_valid.artist == song_df_valid.artist_name,
                                     log_df_valid.song == song_df_valid.title,
                                     log_df_valid.length == song_df_valid.duration]) \
    .select("datetime", "userId", "level", "song_id", "artist_id", "sessionId", "userAgent") \
    .withColumn("songplay_id", monotonically_increasing_id()) \
    .withColumn("year", year("datetime")) \
    .withColumn("month", month("datetime")) \
    .withColumnRenamed("datetime", "start_time") \
    .withColumnRenamed("userId", "user_id") \
    .withColumnRenamed("sessionId", "session_id") \
    .withColumnRenamed("userAgent", "user_agent")

In [32]:
songplays_table.take(5)

[Row(start_time=datetime.datetime(2018, 11, 21, 22, 56, 47, 796000), user_id='15', level='paid', song_id='SOZCTXZ12AB0182364', artist_id='AR5KOSW1187FB35FF4', session_id=818, user_agent='"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/36.0.1985.125 Chrome/36.0.1985.125 Safari/537.36"', songplay_id=0, year=2018, month=11)]

In [33]:
# write songplays table to parquet files partitioned by year and month
songplays_table.write.partitionBy("year", "month").parquet("output/songplays_out/")

In [34]:
songplays_table.join(time_table, songplays_table.start_time == time_table.start_time) \
    .groupBy('weekday') \
    .agg({'weekday':'count'}) \
    .show()

+-------+--------------+
|weekday|count(weekday)|
+-------+--------------+
|    Wed|             1|
+-------+--------------+

